In [1]:
import os

print("Current working directory:", os.getcwd())

Current working directory: c:\Users\Babak.Baradaranhezav\ml-projects\ml_airbnb_price_regression\notebooks


In [2]:
from pathlib import Path
import os

# Set working directory only if not already set
cwd = Path.cwd()
if not (cwd / ".git").exists():
    for parent in cwd.parents:
        if (parent / ".git").exists():
            os.chdir(parent)
            print("Working directory set to repo root:", parent)
            break
    else:
        raise FileNotFoundError("Could not find .git repo root. Are you inside the correct project folder?")
else:
    print("Already in repo root:", cwd)


Working directory set to repo root: c:\Users\Babak.Baradaranhezav\ml-projects\ml_airbnb_price_regression


In [3]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Load cleaned EDA data
df = pd.read_csv('data/processed/cleaned_listings.csv')
print(f"Loaded dataset with shape: {df.shape} which means {df.shape[0]} rows and {df.shape[1]} columns.")

Loaded dataset with shape: (5090, 80) which means 5090 rows and 80 columns.


## Step 1: Feature Engineering – Host Age

**Problem**  
`host_since` is a date column but stored as text. We want to understand how experienced a host is.

**Goal**  
Calculate how long a host has been active (in days).

**Approach**  
Convert `host_since` to datetime, and subtract from today to get `host_age_days`.

In [4]:
# Convert host_since to datetime
if 'host_since' in df.columns:
    df['host_since'] = pd.to_datetime(df['host_since'], errors='coerce')
    df['host_age_days'] = (pd.to_datetime('today') - df['host_since']).dt.days
    print("Added host_age_days 'host_since' converted and 'host_age_days' created.")
else:
    print("'host_since' not found in dataset.")

Added host_age_days 'host_since' converted and 'host_age_days' created.


## Step 2: Feature Engineering – First Review & Categorical Encoding

**Problem**  
- `last_review` is a date stored as text.
- Categorical features like `room_type` must be encoded.

**Goal**  
Convert `last_review` into `days_since_last_review` and one-hot encode selected categorical variables.

**Approach**  
- Convert `last_review` to datetime.
- One-hot encode `room_type`.

Feature: Days since last review

In [5]:
if 'last_review' in df.columns:
    df['last_review'] = pd.to_datetime(df['last_review'], errors='coerce')
    df['days_since_last_review'] = (pd.Timestamp('today') - df['last_review']).dt.days
    print("Added 'days_since_last_review'")

Added 'days_since_last_review'


Feature: Days since last review

In [6]:
if 'room_type' in df.columns:
    room_dummies = pd.get_dummies(df['room_type'], prefix='room', drop_first=True)
    df = pd.concat([df, room_dummies], axis=1)
    print("One-hot encoded 'room_type'")


One-hot encoded 'room_type'


### Step 3: Feature Engineering – Review Scores

**Problem**  
Many review score columns (like `review_scores_rating`, `review_scores_accuracy`, etc.) are numeric but may have missing values.

**Goal**  
- Understand how well-rated each listing is.
- Create an aggregate score or handle missing scores properly.

**Approach**  
- Identify all review score columns.
- Fill missing values with column means or flags.
- (Optional) Create an average score column.


Feature: Review scores & average

In [7]:
# Identify review score columns
review_cols = [col for col in df.columns if col.startswith('review_scores_')]
print(f"Found review score columns: {review_cols}")

# Fill missing values with column means
for col in review_cols:
    if df[col].dtype in ['float64', 'int64']:
        df[col] = df[col].fillna(df[col].mean())
        print(f"Filled missing values in '{col}' with column mean.")

# Optional: Create a total or average score
if review_cols:
    df['avg_review_score'] = df[review_cols].mean(axis=1)
    print("Filled missing review scores and added 'avg_review_score'")


Found review score columns: ['review_scores_rating', 'review_scores_accuracy', 'review_scores_cleanliness', 'review_scores_checkin', 'review_scores_communication', 'review_scores_location', 'review_scores_value']
Filled missing values in 'review_scores_rating' with column mean.
Filled missing values in 'review_scores_accuracy' with column mean.
Filled missing values in 'review_scores_cleanliness' with column mean.
Filled missing values in 'review_scores_checkin' with column mean.
Filled missing values in 'review_scores_communication' with column mean.
Filled missing values in 'review_scores_location' with column mean.
Filled missing values in 'review_scores_value' with column mean.
Filled missing review scores and added 'avg_review_score'


Feature: Host trust signals

In [8]:
host_flags = {
    'host_has_profile_pic': 'host_has_profile_pic',
    'host_identity_verified': 'host_identity_verified'
}

for col, new_col in host_flags.items():
    if col in df.columns:
        df[new_col] = df[col].map({'t': 1, 'f': 0})
        print(f"Converted {col} to binary")


Converted host_has_profile_pic to binary
Converted host_identity_verified to binary


Feature: Extract selected amenities into binary columns

In [9]:
# Only run if amenities column exists
if 'amenities' in df.columns:
    df['amenities_cleaned'] = df['amenities'].str.replace(r"[{}\"]", "", regex=True)
    df['amenities_list'] = df['amenities_cleaned'].str.lower().str.split(",")

    def has_amenity(amenity):
        return df['amenities_list'].apply(lambda x: int(amenity in [a.strip() for a in x]) if isinstance(x, list) else 0)

    important_amenities = {
        'amenity_ac': 'air conditioning',
        'amenity_parking': 'parking',
        'amenity_wifi': 'wifi',
        'amenity_kitchen': 'kitchen',
        'amenity_pets': 'pet(s)',
        'amenity_microwave': 'microwave',
        'amenity_refrigerator': 'refrigerator',
        'amenity_heating': 'heating',
        'amenity_hairdryer': 'hair dryer',
        'amenity_iron': 'iron',
        'amenity_tv': 'tv',
        'amenity_dishwasher': 'dishwasher',
        'amenity_washer': 'washer',
        'amenity_dryer': 'dryer'
    }

    for col, keyword in important_amenities.items():
        df[col] = has_amenity(keyword)
        print(f"Extracted binary column: {col}")
else:
    print("No amenities column found")


Extracted binary column: amenity_ac
Extracted binary column: amenity_parking
Extracted binary column: amenity_wifi
Extracted binary column: amenity_kitchen
Extracted binary column: amenity_pets
Extracted binary column: amenity_microwave
Extracted binary column: amenity_refrigerator
Extracted binary column: amenity_heating
Extracted binary column: amenity_hairdryer
Extracted binary column: amenity_iron
Extracted binary column: amenity_tv
Extracted binary column: amenity_dishwasher
Extracted binary column: amenity_washer
Extracted binary column: amenity_dryer


Drop raw columns

In [10]:
cols_to_drop = ['amenities_cleaned', 'amenities_list', 'amenities']
df.drop(columns=[col for col in cols_to_drop if col in df.columns], inplace=True)


### Step 5: Save Feature-Engineered Dataset

In [11]:
# Save processed dataset for modeling
output_path = Path("data/processed/featured_listings.csv")
df.to_csv(output_path, index=False)
print(f"Saved feature-engineered dataset to {output_path}")


Saved feature-engineered dataset to data\processed\featured_listings.csv
